In [4]:
import pandas as pd
import os
from dotenv import load_dotenv

load_dotenv()

path_to_dumps = os.environ['DATAPATH']

In [8]:
dumps_filenames = os.listdir(path_to_dumps)
dump_example = pd.read_parquet(f'{path_to_dumps}/{dumps_filenames[0]}')

In [12]:
dump_example

,id,url,title,text
0,12,https://en.wikipedia.org/wiki/Anarchism,Anarchism,Anarchism is a political philosophy and moveme...
1,39,https://en.wikipedia.org/wiki/Albedo,Albedo,Albedo (; ) is the fraction of sunlight that i...
2,290,https://en.wikipedia.org/wiki/A,A,"A, or a, is the first letter and the first vow..."
3,303,https://en.wikipedia.org/wiki/Alabama,Alabama,Alabama () is a state in the Southeastern regi...
4,305,https://en.wikipedia.org/wiki/Achilles,Achilles,"In Greek mythology, Achilles ( ) or Achilleus ..."
...,...,...,...,...
156284,5275580,https://en.wikipedia.org/wiki/Brookins%20Campbell,Brookins Campbell,"Brookins Campbell (1808December 25, 1853) was ..."
156285,5275584,https://en.wikipedia.org/wiki/Antonio%20Valencia,Antonio Valencia,"Luis Antonio Valencia Mosquera, known as Anton..."
156286,5275590,https://en.wikipedia.org/wiki/Geraint%20Bowen,Geraint Bowen,Geraint Bowen may refer to:\n\nGeraint Bowen (...
156287,5275592,https://en.wikipedia.org/wiki/Trstenik%20Airport,Trstenik Airport,"Trstenik Airport (, Latin: Aerodrom Trstenik)..."


so the data has the URL, the title and the text. my initial reaction is that the title and url are probably largely irrelevant. there is a few different considerations we'll have to take regarding the text. my main thoughts are
1. information on articles should be relatively interconnected (that's why things like wiki races are a thing!)
2. so the network should be able to gain general information about, ships and guns, for example, and then possibly infer information about warships (given warships is held out as a validation set)
3. the network should be able to learn the structure of english fairly easily using wikipedia.

for number 3 to be true, we might need to do some cleaning of the existing text, as some of the above examples contain some artifacts.

In [15]:
print(dump_example.iloc[1]['text'][0:100])

Albedo (; ) is the fraction of sunlight that is diffusely reflected by a body. It is measured on a s


so these artifacts come from the pronunciation stuff thats at the start of the page. claude can write me a fairly good cleaning function using some fuzzy matching to get rid of most of the artifacts that exist. some other examples are below.

In [19]:
print(dump_example.iloc[1]['text'][-1000:-1])

ry surface to a depth of a meter or so (a few wavelengths of the radar wavelength which is typically at the decimeter scale) using the following empirical relationships:

.

History 
The term albedo was introduced into optics by Johann Heinrich Lambert in his 1760 work Photometria.

See also

 Cool roof
 Daisyworld
 Emissivity
 Exitance
 Global dimming
 Ice–albedo feedback
 Irradiance
 Kirchhoff's law of thermal radiation
 Opposition surge
 Polar see-saw
 Radar astronomy
 Solar radiation management

References

External links

 Albedo Project 
 Albedo – Encyclopedia of Earth
 NASA MODIS BRDF/albedo product site
 Ocean surface albedo look-up-table
 Surface albedo derived from Meteosat observations
 A discussion of Lunar albedos
 reflectivity of metals (chart) 

Land surface effects on climate
Climate change feedbacks
Climate forcing
Climatology
Electromagnetic radiation
Meteorological quantities
Radiometry
Scattering, absorption and radiative transfer (optics)
Radiation
1760s neologism


at the bottom of wikipedia pages there are 'see also' sections which need to be filtered. the below function will be added in `src/askwiki/data/utils/clean.py`

In [ ]:
import re

TAIL_SECTIONS = re.compile(
    r'\n\s*(See also|References|External links|Further reading|Notes|Footnotes|'
    r'Citations|Sources|Bibliography|Notes and references)\s*\n.*', re.S)
EMPTY_PARENS = re.compile(r'\(\s*(?:[;,]\s*)*\)')
LEAD_PUNCT_PAREN = re.compile(r'\(\s*(?:[;,]\s*)+')
TRAIL_PUNCT_PAREN = re.compile(r'(?:\s*[;,])+\s*\)')
MONTHS = 'January|February|March|April|May|June|July|August|September|October|November|December'
GLUED_DATE = re.compile(rf'(\d)({MONTHS})')
TEMPLATES = re.compile(r'\{\{[^{}]*\}\}')
HTML = re.compile(r'<ref[^>]*/>|<ref[^>]*>.*?</ref>|<[^>]+>', re.S)
TABLE_ROWS = re.compile(r'^\s*(\{\||\|\}|\|-|\|).*$', re.M)
MULTISPACE = re.compile(r'[ \t]{2,}')
SPACE_PUNCT = re.compile(r' +([,.;:])')
BLANK_LINES = re.compile(r'\n{3,}')


def clean_text(text: str) -> str:
    text = TAIL_SECTIONS.sub('', text)
    text = HTML.sub('', text)
    for _ in range(3):
        text = TEMPLATES.sub('', text)
    text = TABLE_ROWS.sub('', text)
    text = text.replace("'''", '').replace("''", '')
    text = LEAD_PUNCT_PAREN.sub('(', text)
    text = TRAIL_PUNCT_PAREN.sub(')', text)
    text = EMPTY_PARENS.sub('', text)
    text = GLUED_DATE.sub(r'\1 – \2', text)
    text = MULTISPACE.sub(' ', text)
    text = SPACE_PUNCT.sub(r'\1', text)
    text = re.sub(r'[ \t]+\n', '\n', text)
    text = re.sub(r'\n[ \t]+', '\n', text)
    text = BLANK_LINES.sub('\n\n', text)
    return text.strip()

SyntaxError: unterminated string literal (detected at line 25) (1219884651.py, line 25)